# Fault Tolerance and Checkpointing

Structured Streaming allows us to recover an application by just restarting it.
To do this, we must configure the application to use checkpointing and write-ahead logs, both
of which are handled automatically by the engine.

**Checkpoint** - Structured Streaming periodically save all relevant progress information  as well as the current intermediate
state values to the checkpoint location
Automatically recover its state and start processing data where it left off.
We do not have to manually manage this state on behalf of the application.Structured Streaming does it for us




```
static = spark.read.json("/data/activity-data")

streaming = spark\
			.readStream\
			.schema(static.schema)\
			.option("maxFilesPerTrigger", 10)\
			.json("/data/activity-data")\
			.groupBy("gt")\
			.count()

query = streaming\
		.writeStream\
		.outputMode("complete")\
		.option("checkpointLocation", "/some/python/location/")\
		.queryName("test_python_stream")\
		.format("memory")\
		.start()
```



# Sizing and Rescaling Application

If system is receiving data faster than it can process it, we need more computing power. We can scale up by adding more executors or resources,
and scale down later by removing them. Just be aware that changing the cluster size can temporarily slow things down while Spark reshuffles data.
In the end, it’s a business decision whether you want automatic scaling or just adjust resources manually.

# Metrics and Monitoring

There are 2 key APIs that checks whether the stream is behaving as expected

1. Query Status
2. Recent Progress



```
query = (
    streamingDF
        .writeStream
        .format("memory")
        .queryName("test_stream")
        .outputMode("append")
        .start()
)

progress = query.lastProgress
print(progress)

all_progress = query.recentProgress
for p in all_progress:
  p = query.lastProgress
  print("Input rows:", p["numInputRows"])
  print("Processing speed:", p["processedRowsPerSecond"])
  print("Batch duration (ms):", p["durationMs"]["triggerExecution"])
```



1. Query Status : running the command query.status
{
}
"message" : "Getting offsets from ...",
"isDataAvailable" : true,
"

Recommend :  use the richer StreamingQueryListener API described later to listen to more events.

2. Recent Progress
query.recentProgress --
```
Array({
  "id" : "d9b5eac5-2b27-4655-8dd3-4be626b1b59b",
  "runId" : "f8da8bc7-5d0a-4554-880d-d21fe43b983d",
  "name" : "test_stream",
  "timestamp" : "2017-08-06T21:11:21.141Z",
  "numInputRows" : 780119,
  "processedRowsPerSecond" : 19779.89350912779,
  "durationMs" : {
    "addBatch" : 38179,
    "getBatch" : 235,
    "getOffset" : 518,
    "queryPlanning" : 138,
    "triggerExecution" : 39440,
    "walCommit" : 312
  },
  "stateOperators" : [ {
    "numRowsTotal" : 7,
    "numRowsUpdated" : 7
  } ],
  "sources" : [ {
    "description" : "FileStreamSource[/some/stream/source/]",
    "startOffset" : null,
    "endOffset" : {
      "logOffset" : 0
    },
    "numInputRows" : 780119,
    "processedRowsPerSecond" : 19779.89350912779
  } ],
  "sink" : {
    "description" : "MemorySink"
  }
})

Input rate ≈ numInputRows / triggerDuration
  "numInputRows" : 780119,
  "triggerExecution" : 39440
  "processedRowsPerSecond"(processingrate) : 19779.89350912779
  
  Input Rate = 780119/39.440 ≈ 19,800 rows/sec
  
Note : When input rate is much greater than the processing rate- the stream is falling
behind and you will need to scale the cluster up to handle the larger load.

Let say we have:
.trigger(processingTime="1 minute") this means, Spark will attempt to run a batch every 1 minute.

Actual batch duration is max(triggerExecution, processing time)
so if processing takes 40 seconds, the batch duration will become 40 seconds though we set  1 minute
on the other hand, If  processing takes 90 seconds, the batch duration becomes 90 seconds — even if the trigger is 1 minute
  
```


**Advanced Monitoring with the Streaming Listener**

The StreamingQueryListener class will allow to receive asynchronous updates from the
streaming query in order to automatically output this information to other systems and
implement robust monitoring and alerting mechanisms